# SQL Murder Mystery

## Can you find out whodunnit?

![A decorative illustration of a detective looking at an evidence board.](https://mystery.knightlab.com/174092-clue-illustration.png)

There's been a Murder in SQL City! The SQL Murder Mystery is designed to be both a self-directed lesson to learn SQL concepts and commands and a fun game for experienced SQL users to solve an intriguing crime.

## SQL sleuths start here

A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a **​murder​**that occurred sometime on ​**Jan.15, 2018​** and that it took place in ​**SQL City​**. Start by retrieving the corresponding crime scene report from the police department’s database.

### Exploring the Database Structure

Experienced SQL users can often use database queries to infer the structure of a database. But each database system has different ways of managing this information. The SQL Murder Mystery is built using SQLite. Use this SQL command to find the tables in the Murder Mystery database.

Run this query to find the names of the tables in this database.

SQLite-specific tip: the `sqlite_master` table is SQLite's catalog of tables and schemas, and other databases expose metadata differently.

This command is specific to SQLite. For other databases, you'll have to learn their specific syntax.


In [38]:
# Install required packages
%pip install jupysql sqlalchemy pandas --quiet

# Load SQL magic
%load_ext sql

# Connect to the database
%sql sqlite:///sql-murder-mystery.db
#%config SqlMagic.style = 'table'

Note: you may need to restart the kernel to use updated packages.
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [39]:
%%sql
SELECT name
FROM sqlite_master

Running query in 'sqlite:///sql-murder-mystery.db'

name
crime_scene_report
drivers_license
facebook_event_checkin
interview
get_fit_now_member
sqlite_autoindex_get_fit_now_member_1
get_fit_now_check_in
solution
check_solution
income



Besides knowing the table names, you need to know how each table is structured. The way this works is also dependent upon which database technology you use. Here's how you do it with SQLite.

Run this query to find the structure of the `crime_scene_report` table

Change the value of 'name' to see the structure of the other tables you learned about with the previous query.


In [40]:
%%sql
SELECT sql
FROM sqlite_master
where name = 'crime_scene_report'

Running query in 'sqlite:///sql-murder-mystery.db'

sql
"CREATE TABLE crime_scene_report ( date integer, type text, description text, city text )"



### The rest is up to you!

If you're really comfortable with SQL, you can probably get it from here. To help, here is the schema diagram:

![schema diagram](schema.png)

Use your knowledge of the database schema and SQL commands to find out who committed the murder.
### Check your solution

Did you find the killer? When you think you know the answer, submit your suspect using the following code and find out if you're right.


In [41]:
%%sql
INSERT INTO solution VALUES (1, 'Jeremy Bowers');
SELECT value FROM solution;

Running query in 'sqlite:///sql-murder-mystery.db'

1 rows affected.

## Now, I will solve who the murderer is 
## I will obtain SQL City's crime report that was recorded in January 15, 2018


In [42]:
%%sql
SELECT *
FROM crime_scene_report 
WHERE crime_scene_report.date = 20180115 AND crime_scene_report.city = "SQL City"

Running query in 'sqlite:///sql-murder-mystery.db'

date,type,description,city
20180115,assault,"Hamilton: Lee, do you yield? Burr: You shot him in the side! Yes he yields!",SQL City
20180115,assault,Report Not Found,SQL City
20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


# Obtain further information about 2 witnesses

In [43]:
%%sql
-- Searching for information for first witness who lives at last house on Northwestern Drive
SELECT *
FROM person
WHERE person.address_street_name LIKE "%Northwestern Dr%"
ORDER BY person.address_number DESC
LIMIT 1

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [44]:
%%sql
-- Searching for information about second witness named Annabel who lives on Franklin Ave
SELECT *
FROM person
WHERE person.name LIKE "%Annabel%" AND person.address_street_name LIKE "%Franklin Ave%"

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
16371,Annabel Miller,490173,103,Franklin Ave,318771143


# Obtain interviews from 2 witnesses

In [45]:
%%sql
SELECT *
FROM interview
WHERE interview.person_id = "14887" OR interview.person_id = "16371"

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
14887,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
16371,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


# Some information about murderer
- membership number starts with "48Z"
- is gold member
- license plate contains "H42W"
- worked out last week on January 9th

In [46]:
%%sql
SELECT get_fit_now_member.id, get_fit_now_member.person_id, get_fit_now_member.name, get_fit_now_member.membership_status, person.license_id, drivers_license.plate_number
FROM get_fit_now_member 
    LEFT JOIN person
    ON get_fit_now_member.person_id = person.id
    LEFT JOIN drivers_license
    ON person.license_id = drivers_license.id
WHERE get_fit_now_member.id LIKE "48Z%" AND get_fit_now_member.membership_status = "gold" AND drivers_license.plate_number LIKE "%H42W%"

Running query in 'sqlite:///sql-murder-mystery.db'

id,person_id,name,membership_status,license_id,plate_number
48Z55,67318,Jeremy Bowers,gold,423327,0H42W2


# I will now confirm whether Jeremy Bowers worked out on January 9th, 2018 or not

In [47]:
%%sql
SELECT *
FROM get_fit_now_check_in
WHERE get_fit_now_check_in.membership_id = "48Z55" AND get_fit_now_check_in.check_in_date = "20180109"

Running query in 'sqlite:///sql-murder-mystery.db'

membership_id,check_in_date,check_in_time,check_out_time
48Z55,20180109,1530,1700


# Jeremy Bowers did work out on January 9th, 2018. This means he is the murderer